In [ ]:
# Finding a Protein Motif
"""
This this problem Rosalind is asking to find positions of motifs on publically available data bases, which can change over time.
Here the FASTA files are being validated before, they are compared to remove incorrect or fractutured input.

The three stage pipeline is:
1. Network - get raw text
2. Data Validation - Check that the files are FASTA
3. Biological computation - Scan for the motifs

Without data validation the borken API responces enter the motif search, which tamper with the results.
"""

"""
In this problem Rosalind requires you to extract protein sequences from extermal databases that may change over time, or return
inconsis
"""

In [2]:
import requests

BASE_URL = "https://rest.uniprot.org/uniprotkb/{}.fasta"


def get_sequence(raw_id):
    """
    Robust UniProt fetch:
    - handles secondary IDs (splits '_')
    - follows redirects safely
    - validates FASTA response
    - retries once if needed
    """

    protein_id = raw_id.split('_')[0]

    url = BASE_URL.format(protein_id)

    try:
        response = requests.get(url, allow_redirects=True, timeout=10)
    except requests.RequestException:
        return ""

    # If request failed
    if response.status_code != 200:
        return ""

    fasta = response.text.strip()

    # Must be FASTA
    if not fasta.startswith(">"):
        return ""

    # Extract sequence safely
    lines = fasta.splitlines()
    seq = "".join(line.strip() for line in lines if not line.startswith(">"))

    return seq


def matches_motif(fragment):
    return (
        fragment[0] == 'N'
        and fragment[1] != 'P'
        and fragment[2] in ('S', 'T')
        and fragment[3] != 'P'
    )


def find_motif_positions(sequence):
    positions = []

    for i in range(len(sequence) - 3):
        if matches_motif(sequence[i:i+4]):
            positions.append(i + 1)  # 1-based indexing

    return positions


# ---- INPUT ----
# Chance User_Name to your computer's User name
# This will read the file you downloaded from Rosalind
with open(r"C:\Users\scien\Downloads\rosalind_mprt.txt") as f:
    protein_ids = [line.strip() for line in f if line.strip()]


# ---- PROCESS ----
for pid in protein_ids:

    seq = get_sequence(pid)

    if not seq:
        continue

    pos = find_motif_positions(seq)

    if pos:
        print(pid)
        print(*pos)

P28653_PGS1_MOUSE
271 312
P23185
71 77
P01047_KNL2_BOVIN
47 87 168 169 197 204 280
A8GP89
101
P36912_EBA2_FLAME
5 48 236 278
Q8CE94
369
B5FPF2
18
P39873_RNBR_BOVIN
88
P01866_GCB_MOUSE
185
P01374_TNFB_HUMAN
96
